# 04 - RAG: hybrid retrieval, RRF, cross-encoder rerank, cited answers
Rubric deliverable 3. Assumes Qdrant is running (`localhost:6333`).

In [1]:
import os
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == 'notebooks' else None
os.environ.setdefault('TQDM_DISABLE', '1')
os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS', '1')

'1'

## 1. Chunk the corpus and build the vector index

In [2]:
from src.rag.chunk import build_chunks
from src.rag.index import build_index, index_size
chunks = build_chunks()
print(len(chunks), 'chunks from', len({c.doc_id for c in chunks}), 'documents')
print('indexed:', build_index(chunks, recreate=True), 'points | collection size', index_size())

22 chunks from 7 documents


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

indexed: 22 points | collection size 22


## 2. Hybrid search = dense (Qdrant) + BM25, fused with Reciprocal Rank Fusion

In [3]:
from src.rag.search import hybrid_search
hits = hybrid_search('When should sepsis screening be started?')
for h in hits[:8]:
    print(f'{h.chunk_id:<26} rrf={h.rrf_score:.5f}  dense_rank={h.dense_rank}  bm25_rank={h.bm25_rank}')

altered-consciousness::2   rrf=0.03126  dense_rank=5  bm25_rank=3
sepsis-screening::0        rrf=0.03110  dense_rank=1  bm25_rank=8
temperature-derangement::1 rrf=0.03105  dense_rank=2  bm25_rank=7
tachycardia-af-rvr::2      rrf=0.03048  dense_rank=11  bm25_rank=1
temperature-derangement::0 rrf=0.02986  dense_rank=8  bm25_rank=6
altered-consciousness::0   rrf=0.02964  dense_rank=14  bm25_rank=2
altered-consciousness::1   rrf=0.02921  dense_rank=7  bm25_rank=10
tachycardia-af-rvr::1      rrf=0.02904  dense_rank=6  bm25_rank=12


## 3. Cross-encoder rerank + grounded answer with citations

In [4]:
from src.rag.answer import answer_question
ans = answer_question('When should sepsis screening be started and what are the first steps?')
print(ans.answer)
print()
for c in ans.citations:
    print(f"  [{c['marker']}] {c['title']} - {c['heading']}  (rerank {c['rerank_score']})")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

A NEWS2 aggregate of 5 or more, or a single parameter scoring 3, in a patient with likely infection should trigger a sepsis screen. [1] Screen any patient with a suspected or confirmed infection who also shows signs of acute illness. [1] A fever with a NEWS2 of 5 or more, rigors, or haemodynamic instability should trigger a sepsis screen. [2] A new fever with signs of acute illness should prompt a search for a source: respiratory, urinary, abdominal, skin and soft tissue, central nervous system, and device or line related. [2]

  [1] Recognising sepsis and the Sepsis Six - When to screen for sepsis  (rerank 0.67)
  [2] Fever and hypothermia in the acutely unwell adult - Fever  (rerank 0.413)
  [3] New tachycardia and fast atrial fibrillation - Stable new atrial fibrillation with rapid ventricular response  (rerank -7.695)
  [4] New confusion and reduced consciousness - Immediate assessment  (rerank -8.623)
  [5] Hypotension and the shocked patient - Initial approach to hypotension  (re

## 4. Off-topic question is refused

In [5]:
bad = answer_question('What time does the hospital gift shop close on Sundays?')
print('refused:', bad.refused)
print(bad.answer)

refused: True
No sufficiently relevant guidance was retrieved for this question.


## 5. Link back to the lakehouse - explain a Gold NEWS2 window with citations

In [6]:
from src.rag.answer import explain_gold_window
try:
    ex = explain_gold_window()
    print('generated query:', ex.retrieval[0]['generated_query'])
    print()
    print(ex.answer)
    print()
    for c in ex.citations:
        print(f"  [{c['marker']}] {c['title']} - {c['heading']}")
except Exception as e:
    print('needs a Gold table (run notebook 02 first):', e)

generated query: NEWS2 13 (high risk), respiratory rate 26, SpO2 89% on oxygen, systolic BP 72, heart rate 102, temperature 38.1. Recommended escalation and immediate management?

SpO2 92-93%: start or increase supplemental oxygen to reach the target range, sit the patient upright, and request clinical review. [1] SpO2 94-95%: recheck the probe and trace, increase observation frequency, and look for a cause (atelectasis, early pneumonia, fluid overload, pain limiting breathing). [1] Consider high-flow nasal oxygen or non-invasive ventilation if hypoxaemia persists despite standard oxygen. [1] NEWS2 aggregates seven parameters into a single score: respiratory rate, oxygen saturation, use of supplemental oxygen, systolic blood pressure, pulse rate, level of consciousness on the ACVPU scale, and temperature. [2]

  [1] Acute hypoxaemia and oxygen therapy targets - Responding to a falling SpO2
  [2] NEWS2 scoring and escalation of care - What NEWS2 measures
  [3] NEWS2 scoring and escalati